# 01 - Exploratory Data Analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('...'))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
pd.set_option('display.max_colwidth', 120)

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/reviews.csv', encoding='utf-8')
print("Shape", df.shape)
print("Columns", df.columns.tolist())
df.head(5)

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
for i in [0, 1, 2, 2789, 18690]:
    print(f"Review {i}")
    print(df['reviewText'].iloc[i])
    print()

## 2. Rating Distribution

In [ ]:
print(df['overall'].value_counts().sort_index())
df['overall'].value_counts().sort_index().plot(
    kind='bar', color='steelblue', edgecolor='black'
)
plt.title("Rating Distribution")
plt.xlabel("Stars")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
total = len(df)
for stars, count in df['overall'].value_counts().sort_index().items():
    print(f"{int(stars)} stars: {count:,}  ({count/total*100:.1f}%)") # type: ignore

## 3. Review Length Analysis

In [ ]:
df['review_len'] = df['reviewText'].fillna('').apply(len)
df['word_count'] = df['reviewText'].fillna('').apply(lambda x: len(x.split()))
print("Character Length: ")
print(df['review_len'].describe())
print()
print("Word count stats: ")
print(df['word_count'].describe())

In [ ]:
df[df['word_count'] < 300]['word_count'].hist(bins=50, color='steelblue', edgecolor='black')
plt.title("Review Word Count Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('overall')['word_count'].mean().plot(kind='bar', color='coral', edgecolor='black')
plt.title("Avg Word Count by Star Rating")
plt.xlabel("Stars")
plt.ylabel("Avg Words")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Text Noise Reduction

In [ ]:
sample = df['reviewText'].dropna().sample(2000, random_state=42)
has_html = sample.str.contains(r'<[^>]+>', regex=True).sum()
has_num = sample.str.contains(r'\\d', regex=True).sum()
has_url = sample.str.contains(r'http', regex=True).sum()

print(f"Reviews with HTML tag: {has_html} ({has_html/20:.1f}%)")
print(f"Reviews with numbers: {has_num} ({has_num/20:.1f}%)")
print(f"Reviews with URLs: {has_url} ({has_url/20:.1f}%)")

## 5. Quick Preprocessing Test

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


STOPWORDS = set(stopwords.words('english'))
NEGATION = {'not', 'no', 'never', 'neither', 'nor', 'hardly', 'scarcely'}
STOPWORDS -= NEGATION
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    return text.strip().lower()

def tokenize(text):
    tokens = word_tokenize(text)

    tokens = [
        t for t in tokens
        if t.isalpha()
        and t not in STOPWORDS
        and len(t) > 2
    ]

    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]

    return tokens

raw = df['reviewText'].iloc[20]
cleaned = clean_text(raw)
tokens = tokenize(cleaned)

print("RAW: "); print(raw[:400])
print("CLEANED: "); print(cleaned[:400])
print("TOKENS: "); print(tokens[:30])


In [ ]:
for i in [0, 5, 15, 50, 200]:
    print(f"--- position {i} ---")
    print(df['reviewText'].iloc[i])
    print()

## 6. Sample Size Decision

In [ ]:
import time
reviews = df["reviewText"].dropna()
for n in [1000, 5000, 10000, 20000]:
    sample = reviews.sample(min(n, len(reviews)), random_state=42)
    start = time.time()
    sample.apply(lambda x: tokenize(clean_text(x)))
    elapsed = time.time() - start
    print(f"{n:>6,} rows -> {elapsed:.1f}  (~{elapsed/n*1000:.2f}ms per review)")

## Findings to carry into preprocessing.py

- **Column name for review text:** `reviewText`
- **Column name for rating:** `overall`
- **Encoding needed:** `utf-8`
- **HTML present:** yes
- **Sample size to uss:** 19809 (~20k)
- **Any domain-specific stopwords to add:** none
- **Any cleaning tweaks noticed:** none